# *<center> The Einzel Lens: solve, fly, and tune </center>*

**Purpose.** Walk the `ion_gym` code path module by module, outside the GUI,
on the simplest shipped device: the round einzel lens (r-z, native
primitives, no external simulator). Each stage is a separate, plain code block,
deliberately kept independent so an optimization loop (CMA-ES or otherwise)
can later call the same stages programmatically: *build geometry → solve
fields → fly ions → score*. The notebook closes with the **same ring stack rebuilt natively in full
3-D**, cross-validated against the r-z solve, and then with an optimizer
tuning the lens voltage to a declared objective.

```
PROVENANCE
  origin   : shipped examples einzel_round_r-z.json and
             einzel_lens_stl_plates_full_3-d.json (values mirrored as
             named parameters below)
  authored : the notebook series
  verified : cells executed end-to-end in-sandbox before delivery
```

| Stage | What happens | Module |
|---|---|---|
| A | Geometry, source, **bounds** as spec **objects** | `ion_gym.io.sim_spec` |
| B | Field solve: per-electrode **bases** + superposition | `ion_gym.physics.sim_build` |
| C | Integration parameters bound to the run | `IntegrationSpec` |
| D | Flight, fates, **ensemble statistics** | `fly` + `ion_gym.physics.stats` |
| E | The optimization hook (a voltage sweep) | — |
| 3-D | The same lens in xyz: three views, solved fields | `stl3d` route |

**Running it.** Launch Jupyter from the repo root (so `import ion_gym`
resolves), or `pip install -e .` once to run from anywhere. Heavy steps print
their own measured time; the only genuinely slow step is the 3-D section's
*first* ion flight, which pays a one-time ~1–2 min numba compile.

> Parameter cells are plain Python on purpose: Panel widgets *may* be layered
> on top to gather these values interactively, but results must always come
> from code that consumes the settings — the notebook works as code, not as
> a hidden GUI.

### The following notebook works one instrument end to end, from a declared geometry to a tuned operating point. Specific topics include:

* Building an electrostatic lens as spec **objects** rather than a file.
* Solving the field once and re-weighting it for any set of voltages.
* Flying an ensemble of ions and reducing it to instrument statistics.
* Reading focus off a voltage sweep, then finding it automatically with an
  optimizer — and proving the answer with independent evidence.

### Conventions used in this document:

* **Units are mm, eV, microseconds and volts** unless a name says
  otherwise (`_mm`, `_ev`, `_us`, `_ns`, `_v`); every parameter carries
  its unit in its name or a trailing comment.
* **CAPITALS are parameters you are meant to change**; lower-case names
  are computed values. Parameters sit in a cell immediately before the
  stage that first uses them.
* **`spec` is the declaration, `model` is the solved field, `fly` is the
  integrator.** Anything read off `model` is what the kernel actually
  flew — display equals solve.
* Equations are numbered (**Eqn #1**, **Eqn #2**, ...) and the code that
  implements one names it.

____

## Stage 0 — global style (parameters live with their stages)

Nothing below is a magic number buried in a function body: the design point
is declared here, once, with units, and every later stage reads these names.
Change a value here and re-run — the stages follow. These values reproduce
the shipped examples.

## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the three cylinders (outer two grounded, centre biased), the equipotentials bulging through the gaps, and five ions converging to a focus downstream — note that the ions leave with the same energy they entered with, which is why an einzel lens focuses without changing the beam energy.

Deck: `examples/einzel_round_r-z.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (charter: stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/einzel_round_r-z.json', banked='panel_einzel.png', height=520)


In [ ]:
import copy
import time
import numpy as np

# --- the spec schema: every subsystem of a simulation is a dataclass ------
from ion_gym.io.sim_spec import (
    SimSpec,           # the whole declaration: geometry+source+integration+...
    GeometrySpec,      # domain, grid pitch, symmetry, electrodes
    ElectrodeSpec,     # one named electrode: shapes + potentials
    ShapeSpec,         # one primitive (rect/ellipse/polygon...) in mm-space
    SourceSpec,        # where ions are born, with what energy
    IntegrationSpec,   # dt, t_max, what gets recorded
    BoundsSpec,        # declared kill planes: where a flight may END
    CollisionSpec,     # background gas (off here: a lens flies in vacuum)
)
from ion_gym.io import paths                       # repo-rooted path policy
from ion_gym.physics.sim_build import build_run, build_needs_solve
from ion_gym.physics.symmetry import SymmetrySpec  # declared, never sniffed
from ion_gym.physics.scene3d import (              # analytic 3-D authoring:
    GeomScene, GridSpec, Electrode, Shape,         # exact CSG solids, no
    Cylinder,                                      # STL, no tessellation
)
from ion_gym.physics.build_scene3d import simspec_from_scene
from IPython.display import display, Markdown   # imported ONCE, here
from ion_gym.physics.stats import (                # the GUI's own reducer —
    compute_stats, stats_markdown,                 # one source of truth for
    auto_transmitted_fate, enabled_planes,         # ensemble statistics
    mz_of_results,
)
from ion_gym.viz.viz_core import (                 # ONE Scene, consumed by
    scene_from_simspec, describe_fates,            # the interactive plotly
    interactive_panel, interactive_panels,                    # renderers (the GUI's own
)                                                  # visual language)

# ---- figure style ---------------------------------------------------------
# Field-panel colormap: any plotly colorscale name. Ones that read well:
# "Magma", "Blackbody", "Jet", "Hot", "Plasma", "Cividis".
# (A display preference only — it changes no computed value.)
COLORMAP = "Magma"

# ---- FIGURE SIZE ----------------------------------------------------------
# Every panel in this notebook takes a `figsize=(width, height)` keyword —
# the same role matplotlib's figsize plays, in PIXELS (a browser's native
# unit). Change FIGSIZE here and re-run, or pass figsize= to any single
# call to size just that figure:
#     interactive_panel(scene, "xy", figsize=(1200, 500))
# Omit figsize entirely and the panel is sized true-scale from the data.
FIGSIZE = (900, 380)      # the r-z cross-section is long and thin

# Every other parameter lives in a small cell IMMEDIATELY BEFORE the
# stage that first uses it — adjust knobs where you are, no scrolling.

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


#### Relevant References

* Electrostatic cylinder lenses II: Three element einzel lenses
    * J. B. Adams and F. H. Read
        * *J. Phys. E: Sci. Instrum.* **5**, 150-155 (1972)
        * doi.org/10.1088/0022-3735/5/2/318
* Electrostatic Lens Systems, 2nd ed.
    * D. W. O. Heddle
        * IOP Publishing (2000)
* Principles of Electron Optics
    * P. W. Hawkes and E. Kasper
        * Academic Press

____

## Stage A — the specification, one subsystem at a time

The GUI loads a finished JSON; here we build the same thing from the spec
**objects**, because that is what an optimizer will mutate. Each subsystem
gets its own cell so the role of every parameter set is visible.

### A.1 — electrodes

An `ElectrodeSpec` is a *named* conductor: a list of `ShapeSpec` primitives
plus what is applied to it. Three things matter here:

* **The shape lives in mm-space, in the declared coordinate system.** We
  declare `coords="rz"` in A.2, so `x_mm` is the *axial* position and `y_mm`
  the *radius*. A rect at `y_mm = 6, height_mm = 1` is therefore a ring from
  r = 6 to 7 mm — the bore is the empty 6 mm below it.
* **`dc` is the whole drive** for this device. RF membership (`rf_groups`)
  stays empty — an einzel is electrostatic. The quadrupole notebook is where
  RF groups appear.
* **`is_grid` stays `False`**: these are solid conductors that intercept
  ions. A grid/mesh electrode (`is_grid=True`) imposes its potential but is
  transparent to flight — declaring that wrongly turns an aperture into a
  wall (a real, gated bug class in this repo).

Outer rings grounded, center ring negative: an ion decelerates into the
lens gap, is focused where it moves slowest (that is where the lens has the
most time to act), and reaccelerates out — the classic einzel. Energy
neutrality is bookkeeping, not luck: an electrostatic field conserves

**Eqn #1**

$$\tfrac{1}{2}mv^2 + q\,\Phi(\mathbf{r}) \;=\; \text{const},$$

and with the entrance and exit both at $\Phi = 0$ the kinetic energy out
must equal the kinetic energy in, whatever happens in between. Stage D
checks this per ion from the recorded channel rather than trusting the
argument.

In [ ]:
# ---- Parameters of the lens (mirrors examples/einzel_round_r-z.json)
#      change to suit your system -----------------------------------------
# Geometry (r-z: x = axial coordinate, y = radius; the solver stores the
# r >= 0 half-plane and the renderer unfolds it for display)
DOMAIN_L_MM   = 52.0   # axial extent of the solve domain
DOMAIN_R_MM   = 8.0    # radial extent of the solved half-plane
PITCH_MM      = 0.1    # grid pitch: mm per grid unit (see Stage A.2)
BORE_R_MM     = 6.0    # electrode inner face -> a 6 mm bore radius
RING_LEN_MM   = 9.5    # axial length of each ring electrode
RING_THK_MM   = 1.0    # radial thickness of each ring
RING_Z0_MM    = (4.0, 14.5, 25.0)   # axial START of entrance / lens / exit
V_ENTRANCE    = 0.0    # V — grounded
V_LENS        = -120.0 # V — the tuned focus for this source (Stage E shows it)
V_EXIT        = 0.0    # V — grounded: equal end potentials = energy-neutral

In [ ]:
electrodes = []
for name, z0, volts in zip(("entrance", "lens", "exit"),
                           RING_Z0_MM,
                           (V_ENTRANCE, V_LENS, V_EXIT)):
    # One rect per ring: starts at axial z0, spans RING_LEN_MM along the
    # axis, and occupies BORE_R_MM..BORE_R_MM+RING_THK_MM radially.
    ring = ShapeSpec("rect", {
        "x_mm": z0,                 # axial start (r-z: x IS the axis)
        "y_mm": BORE_R_MM,          # inner face -> defines the bore
        "width_mm": RING_LEN_MM,    # axial length
        "height_mm": RING_THK_MM,   # radial thickness
    })
    electrodes.append(ElectrodeSpec(
        name=name,       # names travel to figures, fates and statistics
        shapes=[ring],
        dc=volts,        # the applied potential; is_grid defaults False
    ))

for e in electrodes:
    p = e.shapes[0].params
    print(f"{e.name:9s} z = {p['x_mm']:5.1f}..{p['x_mm'] + p['width_mm']:5.1f} mm"
          f"   r = {p['y_mm']:.1f}..{p['y_mm'] + p['height_mm']:.1f} mm"
          f"   {e.dc:+7.1f} V")
gap = RING_Z0_MM[1] - (RING_Z0_MM[0] + RING_LEN_MM)
print(f"ring-to-ring gap: {gap:.1f} mm; bore radius {BORE_R_MM:.1f} mm")

### A.2 — domain, grid and declared symmetry

`GeometrySpec` fixes three coupled choices:

* **The domain** (`width_mm` × `height_mm`). It must be large enough that
  the boundary does not distort the lens field — here the rings end 17 mm
  before the domain edge, and Stage B's equipotentials visibly flatten out
  well before it.
* **The pitch** (`mm_per_gu`) — the accuracy/cost dial. Halving `h` in 2-D
  quadruples the node count (and in 3-D octuples it). At 0.1 mm the 1 mm
  ring thickness is 10 cells: comfortably resolved.
* **The symmetry is DECLARED, never sniffed**: `SymmetrySpec(coords="rz")`
  routes the build to the native cylindrical solver — every axisymmetric
  device costs half a plane, by declaration, not by a heuristic guessing
  from the shapes.

What that solver actually solves is Laplace's equation in cylindrical
coordinates with no azimuthal dependence,

**Eqn #2**

$$\frac{\partial^2 \Phi}{\partial z^2}
  + \frac{1}{r}\frac{\partial}{\partial r}
    \!\left(r\,\frac{\partial \Phi}{\partial r}\right) = 0,$$

on the r ≥ 0 half-plane, with the electrode surfaces as fixed-potential
(Dirichlet) boundaries. The pitch `h` is the finite-difference step of
that equation: it must be small against the sharpest potential curvature
you care about — here the ring edges, where the equipotentials bunch.

In [ ]:
geometry = GeometrySpec(
    width_mm=DOMAIN_L_MM,           # axial extent
    height_mm=DOMAIN_R_MM,          # radial extent (the stored half-plane)
    mm_per_gu=PITCH_MM,             # the accuracy/cost dial
    symmetry=SymmetrySpec(coords="rz"),   # declared -> cylindrical solver
    electrodes=electrodes,
)

# Node grid the solver will actually allocate (node-centred registration:
# a W-mm span at pitch h carries W/h + 1 nodes).
nz = int(round(DOMAIN_L_MM / PITCH_MM)) + 1
nr = int(round(DOMAIN_R_MM / PITCH_MM)) + 1
print(f"domain {DOMAIN_L_MM} x {DOMAIN_R_MM} mm at h = {PITCH_MM} mm "
      f"-> {nz} x {nr} = {nz * nr:,} nodes (r-z half-plane)")

### A.3 — the ion source

`SourceSpec` decides what question the flight answers. This one is chosen to
isolate *the lens*:

* **`distribution="disc"`**: births spread uniformly over a disc of radius
  `r_mm`, normal to the flight axis — a parallel beam probing different
  radii of the lens.
* **Monoenergetic** (`ke_lo == ke_hi`): every ion carries exactly 50 eV, so
  any spread at the exit is *geometric* (spherical aberration), not
  chromatic. Set `ke_lo < ke_hi` later to study chromatic focal shift —
  one-line change, that is the point of the spec object.
* **`seed`** pins the pseudo-random births: the run is reproducible, and a
  voltage sweep (Stage E) compares tunes on the *same* ions.

In [ ]:
# ---- Parameters of the ion source -- change to suit your system ----
# (on-axis disc, monoenergetic — see Stage A.3 for why)
N_IONS   = 12
SRC_Z_MM = 1.5         # birth plane (axial), upstream of the entrance ring
SRC_R_MM = 1.2         # disc radius: a fifth of the bore
KE_EV    = 50.0        # kinetic energy, identical for every ion
MZ       = 100.0       # m/z
SEED     = 0           # fixed seed -> reproducible births

In [ ]:
source = SourceSpec(
    n_ions=N_IONS,
    distribution="disc",     # uniform disc, normal to the axis
    x0_mm=SRC_Z_MM,          # birth plane, upstream of the entrance ring
    r_mm=SRC_R_MM,           # disc radius (1.2 mm inside a 6 mm bore)
    axis="x",                # r-z: the axial coordinate is x
    direction=[1.0, 0.0, 0.0],
    ke_lo=KE_EV, ke_hi=KE_EV,   # equal -> monoenergetic (see note above)
    mz_list=[MZ],
    seed=SEED,               # reproducible births
    tob_span_us=0.0,         # all born at t = 0 (no injection spread)
)
print(f"{source.n_ions} ions: m/z {MZ:g} at {KE_EV:g} eV, "
      f"disc r = {SRC_R_MM} mm at z = {SRC_Z_MM} mm, seed {SEED}")

### A.4 — declared bounds  *(USER-EDITABLE)*

`BoundsSpec` declares the **kill planes**: the places a flight is *allowed*
to end. Each axis has an independent min/max plane with its own on/off
switch. An ion crossing an enabled plane terminates with the *transmitted*
fate; with no planes enabled an ion just "leaves the field box", which is a
vaguer statement (the 3-D section demonstrates exactly that difference).

Two things make bounds first-class here:

* **They are drawn.** Every figure from here on shows each enabled plane as
  a dashed line (the same convention the interactive app uses), so the
  geometry picture also says *where the experiment ends*.
* **They drive the statistics.** Stage D's ensemble reducer groups landings
  by *declared* plane — never by inferring a plane from the landing cloud.

Edit the cell: enable a radial bound (`y_max_on=True, y_max=5.0`) to add a
5 mm aperture stop, or move the exit plane — the figures and statistics
follow.

In [ ]:
# ---- Parameters of the declared bounds -- change to suit your system ----
# (Stage A.4; drawn dashed on every figure)
EXIT_PLANE_MM = 51.0   # axial kill plane: crossing it = transmitted

In [ ]:
bounds = BoundsSpec(
    x_max_on=True, x_max=EXIT_PLANE_MM,   # the axial exit plane
    # y_max_on=True, y_max=5.0,           # <- try it: a 5 mm radial stop
)

# The SAME enumeration the statistics use (one authority, never two):
for label, axis, coord in enabled_planes(SimSpec(geometry=geometry,
                                                 bounds=bounds)):
    print(f"declared bound: {label} (axis {axis}, {coord:g} mm)")

### A.5 — assemble and validate

`SimSpec` is the complete declaration — geometry, source, collisions,
integration, bounds. `validate()` refuses an inconsistent spec *now*, with a
named reason, rather than letting a solver fail obscurely later. Collisions
are explicitly off: a lens is characterized in vacuum, and this repo learned
the hard way that a stray gas operating point can silently destroy a
flight's meaning.

In [ ]:
# ---- Parameters of the integrator -- change to suit your system ----
# (Stage C binds these to the run and sanity-checks them)
DT_NS      = 0.5       # step; DC device -> set by transit resolution, not RF
T_MAX_US   = 8.0       # cap (~1.6x the field-free transit; computed in C)
REC_EVERY  = 8         # record every Nth step
CHANNELS   = ["speed", "ke_ev", "e_field"]   # extra recorded columns

In [ ]:
spec = SimSpec(
    name="round einzel (r-z native, from notebook)",
    geometry=geometry,
    source=source,
    integration=IntegrationSpec(          # placeholder; Stage C binds the
        dt_ns=DT_NS, t_max_us=T_MAX_US,   # real values and explains them
        rec_every=REC_EVERY, record_channels=list(CHANNELS)),
    bounds=bounds,
    collisions=CollisionSpec(enabled=False),   # vacuum, deliberately
)
spec.validate()   # refuse-with-diagnostic beats degrade-quietly
print(f"spec {spec.name!r}: {len(spec.geometry.electrodes)} electrodes, "
      f"{spec.source.n_ions} ions, "
      f"{len(enabled_planes(spec))} declared bound(s), vacuum")

### A.6 — read the containers back: what did we just build?

Every stage above wrote INTO the spec; here the spec reads itself back
out. `ElectrodeSpec.describe()` states how each conductor is
constructed — where its metal comes from (inline shapes vs an STL mesh)
and what drives it (its DC bias plus any drive-group memberships) — and
`SimSpec.describe()` composes those into the whole declaration: domain
and lattice, drives, source position and kinematics, gas, integration
contract, and the declared planes. Everything printed is the declared
field the builders consume, so this printout **is** the solver's input,
not a paraphrase of it. Reading it now, before the solve, is the
cheapest correctness check in the notebook: a wrong voltage, a
misplaced source, or an accidental gas load is visible here in plain
words, before any compute is spent.

In [ ]:
# Each conductor, one line: its metal's origin and its drive.
for el in spec.geometry.electrodes:
    print(el.describe())

# The whole container, sectioned, with units — the declaration the
# builders consume, printed back verbatim.
print()
print(spec.describe())


## Stage B — solving the electric field

`build_run(spec)` dispatches on the declared symmetry (`rz` → the native
cylindrical solver) and returns `(model, fly, col_names, births)`.

The core fact the optimization work will exploit: Laplace's equation is
**linear**, so the solver computes one **basis** potential per electrode —
$\phi_i$ solves the boundary-value problem with electrode $i$ at 1 V and
every other conductor grounded — and the physical field is the
voltage-weighted superposition

**Eqn #3**

$$\Phi(\mathbf{r}) \;=\; \sum_i V_i\,\phi_i(\mathbf{r}),
\qquad \nabla^2\phi_i = 0 .$$

The expensive step — solving each $\phi_i$ — depends on **geometry
alone**; changing a *voltage* just changes the weights $V_i$: a re-weight,
no re-solve. (The same idea is commonly called "fast adjust".)

Where the bases live differs by route, and honesty about it matters: this
r-z route caches bases **in-process** (every voltage change within a session
is a re-weight; a new session re-solves, which at this grid is well under a
second). The **disk** cache (`fa_cache`, keyed on geometry alone) backs the
planar and full-3-D routes, where a solve is expensive enough to keep across
sessions — the 3-D section below leans on it.

The cell measures both paths so the claim is demonstrated, not asserted.

In [ ]:
needs_solve = build_needs_solve(spec)   # honest report: what THIS run does
t0 = time.time()
model, fly, col_names, births = build_run(spec, verbose=True)
t_solve = time.time() - t0
what = ("cold solve + cache store" if needs_solve
        else "cache hit: bases loaded, no solve")
print(f"build_run: {t_solve:.2f} s  ({what})")

# A voltage change is a RE-WEIGHT, not a re-solve — measure it.
spec.geometry.electrodes[1].dc = V_LENS * 0.5     # retune the lens...
t0 = time.time()
build_run(spec)                                    # same geometry -> cached
t_reweight = time.time() - t0
spec.geometry.electrodes[1].dc = V_LENS            # ...restore the tune
model, fly, col_names, births = build_run(spec)
print(f"voltage re-weight: {t_reweight:.3f} s  "
      f"({t_solve / max(t_reweight, 1e-9):.0f}x faster than the build above)")
print("(r-z bases are cached in-process: every voltage change this "
      "session is a re-weight; a fresh session re-solves in about the "
      "time measured above)")

### The solved device

Rendering goes through the sanctioned viz path — `scene_from_simspec` builds
one `Scene` **from the solver's own electrode mask and field arrays**
(display equals solver input: the drawing cannot drift from what was
flown), and `interactive_panel` renders it interactively — zoom, pan, hover — in
the same visual language as the GUI. (`render_mpl`/`report` consume the
identical Scene when a static document figure is needed; two consumers,
one truth.) An axisymmetric r-z model is shown as its **full
cross-section**, unfolded about the axis, axial coordinate horizontal. The
dashed line is the declared exit plane from A.4.

In [ ]:
scene = scene_from_simspec(spec, model, field="phi")
interactive_panel(scene, "xy", colorscale=COLORMAP,
                  figsize=FIGSIZE)   # zoom into the gaps

...and the **field magnitude** $|\mathbf{E}| = |\nabla\Phi|$ of the same
solve. **Why show both:** $\Phi$ is what the solver computes, but the
force on an ion goes as the *gradient* — the $|\mathbf{E}|$ contours show
where ions actually get pushed. For an einzel that is a sharp lesson: the
potential fills the whole lens, but $|\mathbf{E}|$ concentrates in the two
**gaps** between the rings. The lens does not live in the electrodes; it
lives in the gaps.

In [ ]:
interactive_panel(scene_from_simspec(spec, model, field="E"),
                  "xy", colorscale=COLORMAP, figsize=FIGSIZE)

...and the **field magnitude** $|\mathbf{E}| = |\nabla\Phi|$ of the same
solve. **Why show both:** $\Phi$ is what the solver computes, but the
force on an ion goes as the *gradient* — the $|\mathbf{E}|$ contours show
where ions actually get pushed. For an einzel that is a sharp lesson: the
potential fills the whole lens, but $|\mathbf{E}|$ concentrates in the
two **gaps** between the rings. The lens does not live in the electrodes;
it lives in the gaps.

In [ ]:
interactive_panel(scene_from_simspec(spec, model, field="E"),
                  "xy", colorscale=COLORMAP, figsize=FIGSIZE)

## Stage C — binding the integration parameters

Three numbers govern the flight, all on `IntegrationSpec`:

The tracer integrates the electrostatic equation of motion,

**Eqn #4**

$$m\,\frac{d\mathbf{v}}{dt} \;=\; q\,\mathbf{E}
 \;=\; -\,q\,\nabla\Phi,$$

with a fixed-step RK4, sampling $\mathbf{E}$ from the solved grid. The
speed of a $\mathrm{KE}$-eV ion of mass $m$ sets every timescale:
$v = \sqrt{2\,\mathrm{KE}\,e/m}$ — the cell below evaluates it.

* **`dt_ns`** — the fixed RK4 step. A DC device sets `dt` by how finely the
  field gradients and the transit must be resolved. (For an RF device — the
  quadrupole and SLIM notebooks — `dt` must instead finely resolve the RF
  period; that rule arrives with the RF.)
* **`t_max_us`** — the cap. An ion still flying at the cap is a *timeout*,
  which in a confining device is the intended outcome, not a truncation.
  Here every ion should exit well before it.
* **`rec_every`** — record every Nth step; the `record_channels` (speed,
  kinetic energy, |E|) are sampled at recorded steps into trajectory
  columns.

The fly function **binds the integration at build time**, so setting these
is followed by a `build_run` re-bind — which the cache makes essentially
free, as Stage B just measured.

In [ ]:
spec.integration = IntegrationSpec(
    dt_ns=DT_NS,                 # 0.5 ns -> ~10,400 steps across the device
    t_max_us=T_MAX_US,
    rec_every=REC_EVERY,
    record_channels=list(CHANNELS),
)
model, fly, col_names, births = build_run(spec)   # re-bind: bases cached

# Sanity: the cap against the analytic field-free transit.
E_CHARGE_C = 1.602176634e-19    # CODATA elementary charge
AMU_KG     = 1.66053906892e-27  # CODATA atomic mass unit
v_mm_us = ((2.0 * KE_EV * E_CHARGE_C / (MZ * AMU_KG)) ** 0.5) * 1e-3
transit_us = (EXIT_PLANE_MM - SRC_Z_MM) / v_mm_us
print(f"{KE_EV:g} eV at m/z {MZ:g}: v = {v_mm_us:.2f} mm/us "
      f"-> field-free transit ~{transit_us:.1f} us")
print(f"t_max = {T_MAX_US} us ({T_MAX_US / transit_us:.1f}x transit); "
      f"dt = {DT_NS} ns; recording every {REC_EVERY} steps")
print(f"recorded columns: {col_names}")
print(f"births: {births.shape[0]} x [x, y, z, vx, vy, vz, tob] "
      "(mm, mm/us, us)")

## Stage D — flying ions

Why does a symmetric, energy-neutral element focus at all? Near the axis
the potential of an axisymmetric field is fixed entirely by its on-axis
profile $\Phi_0(z)$ (a consequence of the Laplace equation above):

**Eqn #5**

$$\Phi(r,z) \;\approx\; \Phi_0(z) \;-\; \frac{r^2}{4}\,\Phi_0''(z)
\qquad\Rightarrow\qquad
E_r \;=\; \frac{r}{2}\,\Phi_0''(z).$$

The radial field is proportional to $r$ — the definition of lens action —
and changes sign with the curvature $\Phi_0''$, so an einzel alternates
converging and diverging regions. The *net* effect is converging because
the ion crosses the converging regions **slowly** (it has been
decelerated there) and the diverging ones fast: same field, more time.
That asymmetry grows with $|V_{\text{lens}}|/\mathrm{KE}$, which is what
Stage E sweeps.

`fly(i)` integrates ion *i* and returns `(traj, summary)`:

* `traj` — recorded steps; columns are `col_names`. In r-z the `y` column
  is the *signed* transverse coordinate — use `|y|` for the radius.
* `summary` — how the flight ended: a fate `kind`, the time of flight, the
  end coordinates. A fate code alone does not say whether a run went well,
  so `describe_fates` turns the population's endings into sentences.

In [ ]:
trajectories, fates, summaries = [], [], []
t0 = time.time()
for i in range(len(births)):
    tr, summ = fly(i)
    summaries.append(summ)
    if tr is not None and len(tr):
        trajectories.append(tr)
        fates.append(str(summ.get("kind", "")))
print(f"flew {len(trajectories)} ions in {time.time() - t0:.2f} s\n")

for line in describe_fates(spec, model, summaries):
    print(line)

# Per-ion endings. ke_end from the RECORDED channel confirms the einzel is
# energy-neutral ion by ion, rather than asserting it from symmetry.
print(f"\n{'ion':>3s} {'tof_us':>7s} {'|r_end|_mm':>10s} {'ke_end_eV':>9s}")
ke_col = col_names.index("ke_ev")
for i, (tr, s) in enumerate(zip(trajectories, summaries)):
    print(f"{i:3d} {s['tof']:7.3f} {abs(s['r_end']):10.3f} "
          f"{tr[-1, ke_col]:9.2f}")

### Ensemble statistics

Reductions go through `ion_gym.physics.stats` — the **same** `compute_stats`
the GUI's statistics card renders, so a notebook number and an app number
can never disagree. Two conventions it enforces:

* the *transmitted* fate is derived from the spec (`auto_transmitted_fate`:
  a declared bound makes plane-crossing the success fate);
* landings are grouped by **declared** plane (`enabled_planes` — A.4's
  list), never by inferring a plane from a possibly-mixed landing cloud.

One honest blank to expect: the KE column reads *"— (route records no
termination KE)"*. The r-z summary contract does not carry `ke_end` (only
the 3-D route does), and a missing measurement renders as a stated blank —
never as a fabricated `0 ± 0`. The per-ion table above gets its KE from the
*recorded channel* instead, which is why it can show the number the summary
lacks. (Adding `ke_end` to the r-z summary is a flagged follow-up.)

In [ ]:
from IPython.display import Markdown

st = compute_stats(
    summaries,
    transmitted_fate=auto_transmitted_fate(spec),  # bound on -> fate 3
    mz=mz_of_results(spec, len(summaries)),        # per-ion m/z pairing
    planes=enabled_planes(spec),                   # DECLARED planes (A.4)
)
Markdown(stats_markdown(st))

The trajectory overlay uses the same renderer; the fate sentences ride on
the figure as notes, and the declared exit plane is the dashed line the
paths terminate on — the picture states where and why the flights ended.

In [ ]:
scene = scene_from_simspec(spec, model, field="phi",
                           trajs=trajectories, fates=fates)
interactive_panel(scene, "xy", colorscale=COLORMAP,
                  figsize=FIGSIZE)
# hover a path near x = 51 to read the exit spot

## Stage E — the shape the optimization will use

Everything above was: *parameters → spec → solve (cached) → fly → score*.
An optimizer's inner loop is exactly that, and because a voltage change is
a re-weight, a **voltage** search costs milliseconds per point (a *geometry*
search re-solves per point — still one function call, just a slower one).

Demonstration: sweep the lens voltage and score the mean exit-spot radius.
Operating point inline: m/z 100 at 50 eV from an on-axis 1.2 mm disc;
score = mean |r| at the declared x = 51 mm plane.

In [ ]:
V_SWEEP = (-80.0, -120.0, -160.0, -200.0)

from ion_gym.progress import quote, track
import time

print(f"{'V_lens':>8s} {'mean |r_end| mm':>16s}   "
      f"(m/z {MZ:g}, {KE_EV:g} eV, disc r {SRC_R_MM} mm, "
      f"exit plane x = {EXIT_PLANE_MM:g} mm)")

# One sweep point = a cached re-weight + the full ensemble flown. Measure
# the FIRST point, then quote the remainder from that measurement --
# quote() refuses to invent a number, so a measurement has to come first.
sweep_mean_r = {}

def sweep_point(v):
    spec.geometry.electrodes[1].dc = float(v)   # mutate the design point
    m, f, _, b = build_run(spec)                # cached -> re-weight only
    r_ends = [abs(f(i)[1]["r_end"]) for i in range(len(b))]
    sweep_mean_r[v] = sum(r_ends) / len(r_ends)
    print(f"{v:8.0f} {sweep_mean_r[v]:16.3f}")

t0 = time.time()
sweep_point(V_SWEEP[0])
t_point = time.time() - t0
quote("lens sweep, remaining points", n_items=len(V_SWEEP) - 1,
      per_item_s=t_point)
for v in track(V_SWEEP[1:], "V sweep"):
    sweep_point(v)

# The focus claim is COMPUTED, never asserted: find the sweep's minimizer
# and report where the shipped tune actually sits relative to it.
best_v = min(sweep_mean_r, key=sweep_mean_r.get)
spec.geometry.electrodes[1].dc = V_LENS         # restore the design point
model, fly, col_names, births = build_run(spec)
if best_v == V_LENS:
    print(f"\nrestored V_lens = {V_LENS:g} V -- the shipped tune is the "
          f"minimizer of this sweep "
          f"(mean |r_end| {sweep_mean_r[best_v]:.3f} mm)")
else:
    print(f"\nrestored V_lens = {V_LENS:g} V -- NOTE: this sweep's "
          f"minimum sits at {best_v:g} V ({sweep_mean_r[best_v]:.3f} mm "
          f"vs {sweep_mean_r[V_LENS]:.3f} mm at the shipped tune). The "
          f"grid is coarse by design; Stage F searches the interval "
          f"properly.")


## The same lens in full 3-D — three views, solved fields

Everything so far leaned on the declared `rz` symmetry: one solved
half-plane. Here we build the **identical ring stack** — same bore, same
rings, same gaps, same voltages, read from the *same Stage-0 parameters* —
as a native full-3-D solve, and let the two solvers check each other.

The authoring route is `scene3d`: **analytic CSG**, no STL, no
tessellation. A ring is exactly what it is —
`Shape(within=[outer cylinder], notin=[bore cylinder])` — and the
rasterizer stamps the exact solid onto the grid. (The STL route exists for
geometry that only a CAD mesh can express; it arrives with the quadrupole
notebook, whose rods are genuinely STL.)

Three doctrines become visible here:

* **Symmetry is declared, never sniffed** — and it pays. The ring stack has
  mirror symmetry in x and y through the axis, so the grid declares
  `mirror="xy"`: on each declared plane the solver imposes the symmetry
  (Neumann) condition

  $$\left.\frac{\partial \Phi}{\partial n}\right|_{\text{mirror plane}} = 0,$$

  works a quarter of the volume, and unfolds to the canonical
  **[-H, +H] frame** (mirror plane at 0): the axis is at x = y = 0, and
  spec coordinates *are* display coordinates.
* **The disk cache is real on this route.** 3-D bases go to `fa_cache`
  keyed on geometry — the first build solves (seconds at this grid), every
  later session loads.
* **A 3-D instrument is never a single projection**: each panel below is a
  **cut** through the solved field at a stated position.

*Cost honesty:* the **first `fly` call in a session compiles the 3-D numba
kernel (~1–2 min, one-time)**; flights after it are instant. The cells
print their own measured times so you see which you paid.

In [ ]:
# ---- Parameters of the 3-D twin -- change to suit your system ----------
# (the SAME rings, rebuilt natively in xyz)
H3_MM = 0.2            # 3-D pitch (r-z runs 0.1; 3-D pays volume, so 2x
                       # coarser -- the cross-check below quantifies the cost)

In [ ]:
# The SAME design point, read from Stage 0 — one source of truth. Axis note:
# scene3d cylinders are z-aligned, so the 3-D twin transports along z (the
# r-z convention calls the axial coordinate x; same device, named per frame).
n_transverse_3d = int(round(DOMAIN_R_MM / H3_MM)) + 1   # folded transverse quadrant
n_axial_3d  = int(round(DOMAIN_L_MM / H3_MM)) + 1   # axial
grid_3d = GridSpec(
    nx=n_transverse_3d, ny=n_transverse_3d, nz=n_axial_3d, mm_per_gu=H3_MM,
    mirror="xy",          # DECLARED symmetry: solve a quarter, unfold
    overhang="allow",     # rings straddle the fold planes by construction —
)                         # the documented authoring pattern, not an error

rings_3d = []
for i, (name, z0, volts) in enumerate(zip(("entrance", "lens", "exit"),
                                          RING_Z0_MM,
                                          (V_ENTRANCE, V_LENS, V_EXIT)),
                                      start=1):
    ring = Shape(
        # a ring IS outer-cylinder-minus-bore; exact, not tessellated
        within=[Cylinder(cx=0.0, cy=0.0, z=z0 + RING_LEN_MM,
                         r=BORE_R_MM + RING_THK_MM, length=RING_LEN_MM)],
        notin=[Cylinder(cx=0.0, cy=0.0, z=z0 + RING_LEN_MM,
                        r=BORE_R_MM, length=RING_LEN_MM)],
    )
    rings_3d.append(Electrode(index=i, name=name, shapes=[ring],
                            voltage=volts))

scene_def = GeomScene(grid=grid_3d, electrodes=rings_3d, units="mm",
                      name="round einzel — native 3-D rings")
scene_def.check()          # refuse-with-diagnostic before any solving
spec_3d = simspec_from_scene(scene_def,
                           name="round einzel (native 3-D rings)")

# The converter supplies DEFAULTS for everything but geometry — configure
# the rest explicitly, from the SAME Stage-0 names. Collisions first: the
# default is 1 Torr N2, and an unnoticed gas operating point silently
# thermalizes a vacuum flight (a hard-won lesson in this repo).
spec_3d.collisions  = CollisionSpec(enabled=False)
spec_3d.source      = SourceSpec(
    n_ions=N_IONS, distribution="disc",
    x0_mm=0.0, y0_mm=0.0,          # ON the axis: x = y = 0 in the
    z0_mm=SRC_Z_MM,                # canonical mirrored frame
    r_mm=SRC_R_MM, axis="z", direction=[0.0, 0.0, 1.0],
    ke_lo=KE_EV, ke_hi=KE_EV, mz_list=[MZ], seed=SEED, tob_span_us=0.0)
spec_3d.integration = IntegrationSpec(
    dt_ns=DT_NS, t_max_us=T_MAX_US, rec_every=REC_EVERY,
    record_channels=list(CHANNELS))
spec_3d.bounds      = BoundsSpec(z_max_on=True, z_max=EXIT_PLANE_MM)
for label, axis, coord in enabled_planes(spec_3d):
    print(f"declared bound: {label} (axis {axis}, {coord:g} mm)")

needs_solve_3d = build_needs_solve(spec_3d)
t0 = time.time()
model_3d, fly_3d, cols3, births_3d = build_run(spec_3d, verbose=True)
print(f"build: {time.time() - t0:.1f} s "
      f"({'cold solve + disk-cache store' if needs_solve_3d else 'disk-cache hit'})")

Fly the same twelve-ion ensemble and let the two solvers grade each other:
an axisymmetric device solved in r-z at h = 0.1 mm and in full 3-D at
h = 0.2 mm must tell the same story, or one of them is wrong.

In [ ]:
from ion_gym.progress import quote, track
import time

# Cost, quoted UP FRONT (the reader aborts before spending it, not
# after): the first 3-D flight pays the one-time numba kernel compile --
# the ~1-2 min documented above. Flights after it are ms-scale, and are
# re-quoted from a measurement once the compile is paid.
quote("3-D twin: first flight", total_s=90.0,
      detail="one-time 3-D kernel compile dominates (documented ~1-2 "
             "min); the measured value prints when it lands")

trajectories_3d, fates_3d, summaries_3d = [], [], []

def bank_3d(i):
    tr, s = fly_3d(i)
    summaries_3d.append(s)
    if tr is not None and len(tr):
        trajectories_3d.append(tr)
        fates_3d.append(str(s.get("kind", "")))

t0 = time.time()
bank_3d(0)
print(f"first fly: {time.time() - t0:.1f} s (one-time compile paid)")
t0 = time.time()
bank_3d(1)
t_each = time.time() - t0
quote("3-D twin: remaining flights", n_items=len(births_3d) - 2,
      per_item_s=t_each)
for i in track(range(2, len(births_3d)), "3-D flights"):
    bank_3d(i)
for line in describe_fates(spec_3d, model_3d, summaries_3d):
    print(line)

# ---- cross-validation against Stage D (operating point: m/z 100, 50 eV,
# disc r 1.2 mm, exit plane at 51 mm axial) ------------------------------
mean = lambda xs: sum(xs) / len(xs)
tof_rz = mean([s["tof"] for s in summaries])
tof_3d = mean([s["tof"] for s in summaries_3d])
r_rz   = mean([abs(s["r_end"]) for s in summaries])
r_3d   = mean([(s["x_end"] ** 2 + s["y_end"] ** 2) ** 0.5 for s in summaries_3d])
print(f"\nTOF   r-z {tof_rz:.4f} us | 3-D {tof_3d:.4f} us | "
      f"d = {abs(tof_3d - tof_rz) * 1e3:.1f} ns "
      f"({abs(tof_3d - tof_rz) / tof_rz * 100:.2f}%)")
print(f"spot  r-z {r_rz:.3f} mm | 3-D {r_3d:.3f} mm  "
      "(both at focus; the difference is discretization-scale -- "
      "h = 0.1 vs 0.2 mm)")


The cut positions are **derived from the spec, and stated on each panel**:
the transverse `xy` cut at the lens-gap midplane (where the focusing
happens), and the two axial cuts through the beam axis — which the
canonical mirrored frame puts at x = y = 0. The declared z = 51 mm exit
plane draws dashed in the two views that contain the z axis.

In [ ]:
# Cuts DERIVED from the declaration (never bare numbers in the render call).
cut_planes_3d = {
    "xy": 0.5 * (RING_Z0_MM[1] + RING_Z0_MM[1] + RING_LEN_MM),  # lens middle
    "xz": float(spec_3d.source.y0_mm),    # axial cut through the beam axis
    "yz": float(spec_3d.source.x0_mm),    # axial cut through the beam axis
}
scene_3d = scene_from_simspec(spec_3d, model_3d, field="phi",
                            trajs=trajectories_3d, fates=fates_3d, slice_at=cut_planes_3d)
panels_3d = interactive_panels(scene_3d, colorscale=COLORMAP)
# R3: the multi-axis interactive set
for v in ("xy", "xz", "yz"):         # each panel states its own cut
    display(panels_3d[v])

# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


## Export — a spec the GUI loads

`SimSpec.to_json` writes the complete declaration (`to_dict` omits nothing,
so the export is byte-stable and diff-able). Load it in the GUI **by path**
(Load spec → pick the file) so any relative resources resolve; the r-z lens
has none, but the 3-D example's STL directory is exactly such a resource.

In [ ]:
out_path = paths.outputs_dir("einzel_lens_from_notebook.json")
spec.to_json(str(out_path))

# Round-trip check: the exported file reconstructs the identical spec.
back = SimSpec.from_json(str(out_path))
assert back.to_dict() == spec.to_dict(), "export did not round-trip"
print(f"exported: {out_path}")
print("round-trip verified: from_json(export) == spec")
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(back, **DECK_OVERRIDES)


## Stage F — letting the machine tune the lens

Stage E swept one voltage by hand and read the focus off a curve. That
works for one knob. Real instruments have several, and a sweep grid grows
as $N^{\,k}$ — so this stage does the job the way the rest of the toolkit
does it: **declare the objective, let an optimizer search.**

### The objective — what "focused" means, numerically

A beam is focused when the ions arrive close together, so the metric is
the **RMS spot radius at the declared exit plane**,

**Eqn #6**

$$r_{\text{RMS}} \;=\; \sqrt{\frac{1}{N}\sum_{i=1}^{N} r_i^{\,2}},$$

plus a **penalty proportional to the fraction of ions lost**, so a tune
that throws ions away can never win by making the survivors look tidy.

**Why RMS rather than the largest radius:** the maximum is set by a single
ion, so it is noisy and rewards luck. The RMS uses the whole ensemble —
and it is the ensemble that determines an instrument's transmission.

### The search — why a derivative-free method

The objective is the output of a simulation: solve the field, fly an
ensemble, reduce to one number. There is no analytic gradient, finite
differences would be swamped by ensemble noise, and the landscape has flat
penalty plateaus (tunes where everything is lost all score alike) that
defeat naive descent.

`ion_gym.physics.optimize` therefore uses **CMA-ES**, the Covariance
Matrix Adaptation Evolution Strategy. In one sentence: it draws a
population from a Gaussian, keeps the better half, and updates both the
mean **and the shape** of that Gaussian toward the directions that have
been paying off — so it learns the valley's orientation instead of
zig-zagging across it. It is a standard choice for expensive, noisy,
derivative-free objectives of modest dimension.

**References (verified)**

* N. Hansen and A. Ostermeier, "Completely Derandomized Self-Adaptation in
  Evolution Strategies", *Evolutionary Computation* **9**(2), 159-195
  (2001) — CMA-ES itself.
* N. Hansen, "The CMA Evolution Strategy: A Tutorial", arXiv:1604.00772
  (2016) — readable description of the algorithm and its defaults.
* J. B. Adams and F. H. Read, "Electrostatic cylinder lenses II: Three
  element einzel lenses", *J. Phys. E: Sci. Instrum.* **5**, 150-155
  (1972) — focal properties of exactly this lens family.
* D. W. O. Heddle, *Electrostatic Lens Systems*, 2nd ed., IOP Publishing
  (2000) — textbook treatment of einzel focusing and its aberrations.
* P. W. Hawkes and E. Kasper, *Principles of Electron Optics*, Academic
  Press — general reference for paraxial optics and aberrations.

### Why this is affordable

The superposition from Stage B again: changing a voltage is a
**re-weight**, not a re-solve. Each candidate costs one ensemble flight,
not one Laplace solve — which is why dozens of evaluations finish in
seconds.

*(This stage needs the `cma` package: `pip install cma`. Nothing else in
the notebook does, and the toolkit says so plainly if it is missing.)*

In [ ]:
# ---- Parameters of the optimization -- change to suit your system --------
V_LENS_DETUNED = -60.0              # deliberately underfocused start
V_LENS_BOUNDS  = (-200.0, -20.0)    # search interval, volts
OPT_MAX_EVALS  = 40                 # ensemble flights the search may spend
OPT_SEED       = 1                  # CMA-ES seed: the search is reproducible
OPT_METRIC     = "spot_size"        # RMS spot radius + loss penalty

from ion_gym.physics.optimize import OptimizeSpec, optimize
from ion_gym.physics.stats import auto_transmitted_fate

def fly_and_measure(spec_in, label):
    """Fly a spec, report the numbers a tune is judged on, and hand back
    the flight itself so the same tune can be SEEN as well as scored.

    Returns (metrics, model, trajectories, fates) — the last three are
    exactly what scene_from_simspec wants, so the before/after pictures
    below are the SAME flights the table reports, never a re-run that
    might differ."""
    model_i, fly_i, _cols, births_i = build_run(spec_in)
    trajectories_i, fates_i, summaries_i = [], [], []
    for i in range(len(births_i)):
        traj_i, summary_i = fly_i(i)
        summaries_i.append(summary_i)
        if traj_i is not None and len(traj_i):
            trajectories_i.append(traj_i)
            fates_i.append(str(summary_i.get("kind", "")))
    arrived = [s for s in summaries_i
               if s["kind"] == auto_transmitted_fate(spec_in)]
    radii = [abs(s["r_end"]) for s in arrived]
    tofs = [s["tof"] for s in arrived]
    metrics = {
        "V_lens": float(spec_in.geometry.electrodes[1].dc),
        "transmission": len(arrived) / max(len(summaries_i), 1),
        "rms_spot_mm": float(np.sqrt(np.mean(np.square(radii)))),
        "max_r_mm": float(np.max(radii)),
        "tof_spread_ns": float(np.std(tofs)) * 1e3,
    }
    print(f"{label:>20s}: V_lens {metrics['V_lens']:+7.1f} V | "
          f"transmission {metrics['transmission']:4.0%} | "
          f"RMS spot {metrics['rms_spot_mm']*1e3:7.1f} um | "
          f"max |r| {metrics['max_r_mm']*1e3:7.1f} um | "
          f"TOF spread {metrics['tof_spread_ns']:.2f} ns")
    return metrics, model_i, trajectories_i, fates_i

# Work on a COPY of the spec already in memory. (This used to reload
# the file written by the export cell, which quietly made this stage
# depend on that cell having run — a cross-cell coupling through a
# filename. A stage should depend on objects it can see, not on side
# effects elsewhere in the notebook.)
spec_detuned = copy.deepcopy(spec)
spec_detuned.geometry.electrodes[1].dc = V_LENS_DETUNED
before, model_before, trajectories_before, fates_before = \
    fly_and_measure(spec_detuned, "before (detuned)")

The detuned beam arrives wide. Now hand the same spec to the optimizer:
one free parameter (the lens electrode's DC), bounded, with the objective
above.

In [ ]:
search = OptimizeSpec(
    params=["geometry.electrodes[1].dc"],      # a dotted path INTO the spec
    lo=[V_LENS_BOUNDS[0]], hi=[V_LENS_BOUNDS[1]],
    metric=OPT_METRIC, max_evals=OPT_MAX_EVALS, seed=OPT_SEED)

t_opt = time.time()
result = optimize(spec_detuned, search)
V_LENS_BEST = float(result.best_params["geometry.electrodes[1].dc"])
print(f"CMA-ES: {result.n_evals} ensemble flights in "
      f"{time.time()-t_opt:.1f} s")
print(f"  best lens voltage : {V_LENS_BEST:+.2f} V")
print(f"  search objective  : {result.best_value*1e3:.1f} um")
print(f"  fresh-seed audit  : {result.audit_value*1e3:.1f} +/- "
      f"{result.audit_std*1e3:.1f} um")
print("  (the audit re-flies the winner on ensembles the search never saw:"
      " a value that does not reproduce on fresh draws is a lucky sample,"
      " not an optimum)")

spec_optimized = copy.deepcopy(spec)
spec_optimized.geometry.electrodes[1].dc = V_LENS_BEST
after, model_after, trajectories_after, fates_after = \
    fly_and_measure(spec_optimized, "after (optimized)")

In [ ]:
rows = [
    ("lens voltage (V)", f"{before['V_lens']:+.1f}",
     f"{after['V_lens']:+.1f}"),
    ("transmission", f"{before['transmission']:.0%}",
     f"{after['transmission']:.0%}"),
    ("RMS spot (um)", f"{before['rms_spot_mm']*1e3:.1f}",
     f"{after['rms_spot_mm']*1e3:.1f}"),
    ("max |r| (um)", f"{before['max_r_mm']*1e3:.1f}",
     f"{after['max_r_mm']*1e3:.1f}"),
    ("TOF spread (ns)", f"{before['tof_spread_ns']:.2f}",
     f"{after['tof_spread_ns']:.2f}"),
]
lines = ["| metric | before | after |", "|---|---|---|"]
lines += [f"| {name} | {b} | {a} |" for name, b, a in rows]
lines.append(f"\nRMS spot improved "
             f"{before['rms_spot_mm']/after['rms_spot_mm']:.0f}x at the "
             f"declared exit plane (x = {EXIT_PLANE_MM:g} mm), m/z {MZ:g} "
             f"at {KE_EV:g} eV, {N_IONS} ions from a {SRC_R_MM:g} mm disc.")
Markdown("\n".join(lines))

### The same two tunes, flown

The table is the measurement; this is what it looks like. Both panels are
the **same flights** the table scored — same field convention, same
declared exit plane (dashed) as Stage D, so they can be compared directly
with the figures there.

Watch where the crossover sits. Detuned, the beam is still converging
when it reaches the plane — the lens is too weak and its focus lies
somewhere downstream, off the end of the device. Optimized, the crossover
lands **on** the plane, which is all "focused" means here: the objective
does not know about focal length, only about how close together the ions
arrive at the place we declared to care about.

In [ ]:
interactive_panel(
    scene_from_simspec(spec_detuned, model_before, field="phi",
                       trajs=trajectories_before, fates=fates_before),
    "xy", colorscale=COLORMAP, figsize=FIGSIZE)

In [ ]:
interactive_panel(
    scene_from_simspec(spec_optimized, model_after, field="phi",
                       trajs=trajectories_after, fates=fates_after),
    "xy", colorscale=COLORMAP, figsize=FIGSIZE)

### Why *this* voltage — the evidence, not the optimizer's word

An optimizer returning a number is not a result. Two checks make it one.

**1. The landscape.** Sweep the objective across the search interval and
see whether the returned point really sits at the minimum. That is
affordable with one knob — and impossible with five, which is the honest
argument for having an optimizer at all.

**2. The noise floor.** The audit re-flew the winner on fresh ensembles;
its spread is the **resolution of the whole exercise**. Two tunes whose
objectives differ by less than that are not distinguishable, and calling
one better would be reading noise. The shaded band below is that floor.

In [ ]:
# ---- Parameters of the validation sweep -- change to suit your system ----
SWEEP_N      = 25    # points across the FULL interval (the landscape)
FINE_SPAN_V  = 10.0  # +/- volts around the optimum (the tolerance window)
FINE_N       = 21    # points in that window -> 1 V steps

sweep_volts = np.linspace(V_LENS_BOUNDS[0], V_LENS_BOUNDS[1], SWEEP_N)
sweep_rms = []
spec_sweep = copy.deepcopy(spec)
for volts in sweep_volts:
    spec_sweep.geometry.electrodes[1].dc = float(volts)
    model_s, fly_s, _cols_s, births_s = build_run(spec_sweep)  # re-weight
    summaries_s = [fly_s(i)[1] for i in range(len(births_s))]
    radii_s = [abs(s["r_end"]) for s in summaries_s
               if s["kind"] == auto_transmitted_fate(spec_sweep)]
    sweep_rms.append(np.sqrt(np.mean(np.square(radii_s))) if radii_s
                     else np.nan)
sweep_rms = np.asarray(sweep_rms, float)

# A FINE sweep around the optimum. The coarse sweep answers "is the
# optimizer at the minimum?"; it cannot answer "how tightly must the
# supply hold?" because its step is wider than the basin. Two sweeps, two
# questions — stated rather than blurred.
fine_volts = np.linspace(V_LENS_BEST - FINE_SPAN_V,
                         V_LENS_BEST + FINE_SPAN_V, FINE_N)
fine_rms = []
for volts in fine_volts:
    spec_sweep.geometry.electrodes[1].dc = float(volts)
    model_f, fly_f, _cols_f, births_f = build_run(spec_sweep)
    summaries_f = [fly_f(i)[1] for i in range(len(births_f))]
    radii_f = [abs(s["r_end"]) for s in summaries_f
               if s["kind"] == auto_transmitted_fate(spec_sweep)]
    fine_rms.append(np.sqrt(np.mean(np.square(radii_f))) if radii_f
                    else np.nan)
fine_rms = np.asarray(fine_rms, float)

noise_floor = max(float(result.audit_std), 1e-9)
best_sweep = float(np.nanmin(sweep_rms))
indistinguishable = sweep_volts[np.abs(sweep_rms - best_sweep) <= noise_floor]
print(f"sweep minimum        : {best_sweep*1e3:.1f} um at "
      f"{sweep_volts[int(np.nanargmin(sweep_rms))]:+.1f} V")
print(f"optimizer's answer   : {V_LENS_BEST:+.2f} V")
sweep_step = float(sweep_volts[1] - sweep_volts[0])
print(f"sweep step           : {sweep_step:.1f} V "
      f"({SWEEP_N} points across the interval)")
if len(indistinguishable) > 1:
    print(f"tunes indistinguishable from the best (within the "
          f"{noise_floor*1e3:.1f} um floor): "
          f"{indistinguishable.min():+.1f} .. "
          f"{indistinguishable.max():+.1f} V")
else:
    print(f"only ONE swept point ({indistinguishable[0]:+.1f} V) lies within"
          f" the {noise_floor*1e3:.1f} um noise floor -> the true basin is"
          f" NARROWER than the {sweep_step:.1f} V sweep step, so this sweep"
          " cannot measure its width. Refine SWEEP_N to resolve it.")
# The tolerance, measured on the FINE sweep against a STATED criterion
# (spot within 2x the best) rather than an eyeballed "looks flat".
best_fine = float(np.nanmin(fine_rms))
usable = fine_volts[fine_rms <= 2.0 * best_fine]
print(f"fine sweep ({fine_volts[1]-fine_volts[0]:.1f} V steps): best "
      f"{best_fine*1e3:.1f} um at {fine_volts[int(np.nanargmin(fine_rms))]:+.1f} V")
print(f"lens voltages keeping the spot within 2x the best "
      f"({2*best_fine*1e3:.1f} um): {usable.min():+.1f} .. "
      f"{usable.max():+.1f} V -> a {usable.max()-usable.min():.1f} V window, "
      f"{100*(usable.max()-usable.min())/abs(V_LENS_BEST):.1f}% of the "
      "operating voltage. THAT is the supply stability this design needs.")

# Did the optimizer actually land in that window? Reported either way.
V_fine_best = float(fine_volts[int(np.nanargmin(fine_rms))])
offset_v = abs(V_LENS_BEST - V_fine_best)
if usable.min() <= V_LENS_BEST <= usable.max():
    print(f"the CMA-ES answer ({V_LENS_BEST:+.2f} V) lies INSIDE that "
          "window: the search converged to the basin.")
else:
    print(f"the CMA-ES answer ({V_LENS_BEST:+.2f} V) sits {offset_v:.1f} V "
          f"from the fine-sweep minimum ({V_fine_best:+.1f} V), worth "
          f"{(after['rms_spot_mm']-best_fine)*1e3:.1f} um of extra spot.")
    print(f"  Not a bug — a BUDGET statement: the basin is ~"
          f"{usable.max()-usable.min():.0f} V wide inside a "
          f"{V_LENS_BOUNDS[1]-V_LENS_BOUNDS[0]:.0f} V search interval, i.e."
          f" {100*(usable.max()-usable.min())/(V_LENS_BOUNDS[1]-V_LENS_BOUNDS[0]):.1f}%"
          f" of it, and {OPT_MAX_EVALS} evaluations were allowed. Raise"
          " OPT_MAX_EVALS or tighten the bounds and it closes the gap;"
          " that trade — search cost against final precision — is the"
          " thing to internalize.")

import plotly.graph_objects as go
fig_landscape = go.Figure()
fig_landscape.add_scatter(x=sweep_volts, y=sweep_rms*1e3,
                          mode="lines+markers", name="objective (swept)")
fig_landscape.add_scatter(x=fine_volts, y=fine_rms*1e3, mode="lines",
                          name=f"fine sweep (+/-{FINE_SPAN_V:g} V)",
                          line=dict(width=2, dash="dot"))
fig_landscape.add_scatter(x=[V_LENS_BEST], y=[after["rms_spot_mm"]*1e3],
                          mode="markers", name="CMA-ES answer",
                          marker=dict(size=14, symbol="star",
                                      color="#00e5ff"))
fig_landscape.add_scatter(x=[before["V_lens"]],
                          y=[before["rms_spot_mm"]*1e3], mode="markers",
                          name="starting point",
                          marker=dict(size=11, symbol="x", color="#d62728"))
fig_landscape.add_hrect(y0=0.0, y1=(best_sweep + noise_floor)*1e3,
                        fillcolor="#00e5ff", opacity=0.12, line_width=0,
                        annotation_text="within the ensemble noise floor")
fig_landscape.update_layout(
    title=f"objective landscape: RMS spot at x = {EXIT_PLANE_MM:g} mm "
          f"(m/z {MZ:g}, {KE_EV:g} eV)",
    xaxis_title="lens electrode voltage (V)",
    yaxis_title="RMS spot radius (um)", yaxis_type="log",
    width=FIGSIZE[0], height=460,
    margin=dict(l=70, r=20, t=52, b=52))
fig_landscape

### What this result is, and what it is not

* It is the best lens voltage **for this operating point** — m/z, kinetic
  energy, source size and the declared plane all enter the objective. A
  different energy focuses at a different voltage, and that dependence is
  precisely chromatic aberration (Heddle, ch. 5).
* It is **one parameter**. The same `OptimizeSpec` accepts a list, so
  tuning all three electrodes is a two-line change — after which the
  validation sweep above is no longer possible, which is the point.
* The optimum is quoted **with its audit spread**, and the improvement
  **with its operating point**. A spot size without its tune and geometry
  is not a result.

## Where the series goes next

* **Quadrupole** (`quadrupole_stl_rods_full_3-d.json`): the STL route
  (geometry a CAD mesh expresses), RF drive groups — `dt` vs the RF period
  becomes real — and the Mathieu stability picture.
* **Ion funnel**: a DC *ladder* (`DCGroupSpec` deriving a ring gradient
  from two endpoint voltages instead of per-ring magic numbers).
* **SLIM**: travelling-wave RF groups, declared mirror symmetry, and the
  memory budget of an 8M-node domain.

The manual (`docs/ion_gym_user_manual.pdf`) covers the same ground in
prose; this notebook is the executable half.

## Read-out — what this notebook established

- **The lens works by decelerating, then re-accelerating.** The centre electrode's potential does not change an ion's final energy (it enters and leaves the same field-free space), but it does change the *path*: rays bend inward on the way in and again on the way out. That is why an einzel lens focuses at fixed energy — the single most useful property in an ion-optical column.
- **Focal length shortens as |V_centre| rises**, and the shortening is not linear: near the retarding limit the focus runs away toward the electrode and aberration grows. The tuning curve you flew shows both effects; the useful operating range is the shallow part of it.
- **What would have falsified the model:** rays crossing the axis at different points depending on where they started transversely, with no common focus at any voltage. That is spherical aberration dominating, and it is what you would see if the aperture were too large for the electrode spacing.
- **Numbers carry their operating point.** A focal length quoted without its lens voltage, gap, aperture, and ion energy is not a result — every figure here is labelled accordingly.